<a href="https://colab.research.google.com/github/la-becca/Projects/blob/main/Projeto_Analista_de_Credito_com_IA/Projeto_Analista_de_Cr%C3%A9dito_com_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalação de pacotes

In [1]:
!pip install openai scikit-learn pandas Groq kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.1 MB/s eta 0:00:00


# Imports

In [2]:
from google.colab import userdata

In [3]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = userdata.get('API_Key_GROQ')

In [4]:
import os
os.environ['KAGGLE_API_TOKEN'] = userdata.get('API_Kaggle')

In [5]:
import pandas as pd

# Downlod da base

In [6]:
!kaggle competitions download -c home-credit-default-risk

100% 688M/688M [00:06<00:00, 110MB/s]



In [7]:
!unzip home-credit-default-risk.zip

Archive:  home-credit-default-risk.zip
  inflating: HomeCredit_columns_description.csv  
  inflating: POS_CASH_balance.csv    
  inflating: application_test.csv    
  inflating: application_train.csv   
  inflating: bureau.csv              
  inflating: bureau_balance.csv      
  inflating: credit_card_balance.csv  
  inflating: installments_payments.csv  
  inflating: previous_application.csv  
  inflating: sample_submission.csv   


In [8]:
df = pd.read_csv("application_train.csv")
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


# Selecionar variáveis

In [9]:
cols = [
    "TARGET",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "DAYS_EMPLOYED",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

df = df[cols].dropna()
df.head()

,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,DAYS_EMPLOYED,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3
0,1,202500.000,406597.5,24700.5,-637,0.083037,0.262949,0.139376
6,0,171000.000,1560726.0,41301.0,-3130,0.774761,0.724000,0.492060
8,0,112500.000,1019610.0,33826.5,365243,0.587334,0.205747,0.751724
10,0,112500.000,652500.0,21177.0,-679,0.319760,0.651862,0.363945
11,0,38419.155,148365.0,10678.5,365243,0.722044,0.555183,0.652897


# Feature Engineering

* DTI → quanto da renda está comprometida
* credit_income_ratio → tamanho do crédito vs renda

In [10]:
df["dti"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df["credit_income_ratio"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]

# Train

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["TARGET"])
y = df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames, retaining column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

In [12]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

modelLR = LogisticRegression(max_iter=1000)
modelLR.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [13]:
# Random Forest
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

modelRF = RandomForestClassifier(n_estimators=100)
modelRF.fit(X_train, y_train)

RandomForestClassifier()

# Avaliação AUC

In [14]:
from sklearn.metrics import roc_auc_score

predsLR = modelLR.predict_proba(X_test_scaled)[:,1]
print('Regressão Logística: ',roc_auc_score(y_test, predsLR))

predsRF = modelRF.predict_proba(X_test)[:,1]
print('Random Forest: ',roc_auc_score(y_test, predsRF))

Regressão Logística:  0.7434078685142494
Random Forest:  0.7097840509625946


# Teste Cliente Real

In [15]:
cliente = X_test.iloc[[0]]
prob_default = modelLR.predict_proba(cliente)[0][1]
prob_default

np.float64(1.0)

In [16]:
def criar_prompt(cliente, prob):
    return f"""
Você é um analista de crédito de um grande banco.

Dados do cliente:
{cliente.to_dict()}

Probabilidade de inadimplência: {prob:.2f}

Considere boas práticas de crédito.

Responda:
- Classificação de risco
- Principais fatores
- Recomendação
- Justificativa profissional
"""

In [17]:
import groq
import os

client = groq.Groq(api_key=os.environ["GROQ_API_KEY"])

prompt = criar_prompt(cliente, prob_default)

response = client.chat.completions.create(
    messages=[
        {"role": "system", "content": "Você é um especialista em risco de crédito."},
        {"role": "user", "content": prompt}
    ],
    model="llama-3.3-70b-versatile"
)

analise = response.choices[0].message.content
print(analise)

**Classificação de Risco:** Alto Risco

**Principais Fatores:**

1. **Probabilidade de Inadimplência:** A probabilidade de inadimplência calculada é de 1,00, o que indica uma alta probabilidade de o cliente não pagar o crédito.
2. **Relação Dívida-Renda (DTI):** O coeficiente de dívida de 0,5666 indica que o cliente já compromete uma parte significativa de sua renda com pagamentos de dívidas, o que pode dificultar o pagamento de mais uma dívida.
3. **Renda e Crédito:** O valor do crédito (AMT_CREDIT) é significativamente alto em relação à renda total (AMT_INCOME_TOTAL), o que pode ser um indicador de que o cliente pode ter dificuldade em pagar o crédito.
4. **Anuidade:** O valor da anuidade (AMT_ANNUITY) é alto em relação à renda, o que pode aumentar a pressão sobre o orçamento do cliente.

**Recomendação:** Não aprovar o crédito.

**Justificativa Profissional:**

Como analista de crédito, é fundamental considerar boas práticas de crédito para garantir que o risco de crédito seja minim